In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# 
save_dir = 'C:/Users/Administrator/AI/Mask_std/aug'
word = ['0', '1'] 
img_size = 128

X = []
Y = []

print("Loading...")
for idx, label in enumerate(word):
    folder_path = os.path.join(save_dir, label)
    if not os.path.exists(folder_path): continue
    
    files = os.listdir(folder_path)
    for f in files:
        full_path = os.path.join(folder_path, f)
        img = cv2.imread(full_path)
        if img is not None:
            img = cv2.resize(img, (img_size, img_size))
            X.append(img)
            Y.append(idx) # 

X = np.array(X, dtype="float32") / 255.0
Y = np.array(Y)

# 
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f"Data Segmentation: Train {len(X_train)}, Val {len(X_val)}, Test {len(X_test)}")

Loading...
Data Segmentation: Train 322, Val 81, Test 101


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_binary_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(img_size, img_size, 3)),
        MaxPooling2D(2, 2),
        
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Dropout(0.25),
        
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid') 
    ])
    
    model.compile(optimizer='adam', 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

model = build_binary_model()
model.summary()

history = model.fit(X_train, y_train, 
                    epochs=20, 
                    batch_size=32, 
                    validation_data=(X_val, y_val))

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 126, 126, 32)      896       
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 63, 63, 32)        0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 61, 61, 64)        18496     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 30, 30, 64)        0         
_________________________________________________________________
dropout (Dropout)            (None, 30, 30, 64)        0         
_________________________________________________________________
flatten (Flatten)            (None, 57600)             0         
_________________________________________________________________
dense (Dense)                (None, 128)               7

In [4]:
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

score = model.evaluate(X_test, y_test, verbose=0)
print(f"Test (Loss): {score[0]:.4f}")
print(f"Test (Accuracy): {score[1]*100:.2f}%")

#tf.keras.models.load_model("model_trained_data.h5", compile=False)
model.save("model_trained_data.h5")
#keras.saving.save_model("model_trained_data.h5")
print("model_trained_data.h5 Saved")

Test (Loss): 0.0670
Test (Accuracy): 97.03%
model_trained_data.h5 Saved
